# Aarohan-350M — Kaggle SFT (Instruction Fine-Tuning) Notebook

**Run AFTER pre-training is complete.**

This notebook fine-tunes the pre-trained **Aarohan-350M** base model on ~146K instruction-response pairs
so the model learns to answer Software Engineering questions like a chat assistant.

**Instructions:**
1. Enable GPU: Settings → Accelerator → **T4 GPU** (No TPU needed!)
2. Add your pre-trained checkpoint: attach the output of your previous pre-training notebook
3. Add Kaggle Secrets: `WANDB_API_KEY`, `GITHUB_TOKEN`
4. Click **Run All**

> SFT takes ~4-6 hours — fits in a single Kaggle GPU session.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'GPU required! Enable T4 GPU in Settings → Accelerator'

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q wandb tokenizers datasets pyyaml

In [ ]:
# ── Cell 3: Clone Aarohan-350M training code ──────────────────
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_REPO = 'Abhik2005/se-llm-data'

if not os.path.exists('/kaggle/working/se-llm-350m'):
    try:
        token    = secrets.get_secret('GITHUB_TOKEN')
        repo_url = f'https://{token}@github.com/{GITHUB_REPO}.git'
    except Exception:
        repo_url = f'https://github.com/{GITHUB_REPO}.git'
    !git clone {repo_url} /kaggle/working/se-llm-350m
else:
    !git -C /kaggle/working/se-llm-350m pull

%cd /kaggle/working/se-llm-350m
!ls -la

In [ ]:
# ── Cell 4: Generate SFT instruction dataset ──────────────────
# Downloads ~146K coding Q&A pairs from HuggingFace
# Takes ~10-15 minutes. Skip if already generated.
import os

os.makedirs('data/sft', exist_ok=True)

if not os.path.exists('data/sft/sft_data.jsonl'):
    print('Generating SFT dataset...')
    !python data/sft_data.py
else:
    with open('data/sft/sft_data.jsonl') as f:
        n = sum(1 for _ in f)
    size_mb = os.path.getsize('data/sft/sft_data.jsonl') / 1e6
    print(f'SFT dataset already exists: {n:,} samples | {size_mb:.1f} MB')

In [ ]:
# ── Cell 5: Link tokenizer ────────────────────────────────────
import os

os.makedirs('tokenizer', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('checkpoints_sft', exist_ok=True)

# Link tokenizer from Kaggle datasets
tok_candidates = [
    '/kaggle/input/datasets/vedase/se-llm-data/tokenizer.json',
    '/kaggle/input/se-llm-data/tokenizer.json',
]
for tok_src in tok_candidates:
    if os.path.exists(tok_src):
        tok_dest = 'tokenizer/tokenizer.json'
        if not os.path.exists(tok_dest):
            os.symlink(tok_src, tok_dest)
        print(f'Linked tokenizer from {tok_src}')
        break
else:
    print('WARNING: tokenizer.json not found — add the se-llm-data dataset')

In [ ]:
# ── Cell 6: Load pre-trained Aarohan-350M checkpoint ──────────
# Attach the OUTPUT of your previous pre-training notebook as input.
# Kaggle will put it at /kaggle/input/<notebook-slug>/
import os, shutil, glob, torch

# Search all possible locations for best.pt
search_paths = glob.glob('/kaggle/input/**/best.pt', recursive=True)
search_paths += glob.glob('/kaggle/input/**/checkpoints/best.pt', recursive=True)

base_checkpoint = None

if search_paths:
    src = search_paths[0]
    dest = 'checkpoints/pretrain_best.pt'
    if not os.path.exists(dest):
        shutil.copy2(src, dest)
    base_checkpoint = dest
    print(f'Found pre-trained checkpoint: {src}')

    # Show checkpoint info
    ckpt = torch.load(dest, map_location='cpu', weights_only=False)
    print(f'  Model:  {ckpt.get("model_config", {}).get("name", "unknown")}')
    print(f'  Step:   {ckpt.get("step", "unknown"):,}')
    print(f'  Tokens: {ckpt.get("tokens_processed", 0)/1e9:.3f}B')
    print(f'  Val Loss: {ckpt.get("val_loss", 0):.4f}')
else:
    print('WARNING: No pre-trained best.pt found!')
    print('Attach your pre-training notebook output as input data in the sidebar.')

In [ ]:
# ── Cell 7: Login to W&B ──────────────────────────────────────
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets   = UserSecretsClient()
    wandb_key = secrets.get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print('W&B logged in')
except Exception as e:
    print(f'W&B login skipped: {e}')

In [ ]:
# ── Cell 8: RUN INSTRUCTION FINE-TUNING ───────────────────────
# Fine-tunes Aarohan-350M on 146K coding Q&A pairs.
# Teaches the model to answer SE questions like a chat assistant.
# Expected time: ~4-6 hours on T4 GPU.

assert base_checkpoint is not None, 'No pre-trained checkpoint found! Run Cell 6 again.'

cmd = f'python training/sft.py --config configs/350m.yaml --base-checkpoint {base_checkpoint}'
print(f'Running: {cmd}\n')
!{cmd}

In [ ]:
# ── Cell 9: Quick quality test ─────────────────────────────────
# Test Aarohan-350M chat model with sample SE prompts
import torch, os, sys
sys.path.insert(0, '/kaggle/working/se-llm-350m')

from evaluation.generate import load_model_from_checkpoint, load_tokenizer, chat_turn

device    = torch.device('cuda')
ckpt_path = 'checkpoints_sft/sft_final.pt'

# Fall back to latest checkpoint if final not saved yet
if not os.path.exists(ckpt_path):
    import glob
    pts = sorted(glob.glob('checkpoints_sft/*.pt'))
    ckpt_path = pts[-1] if pts else None

if ckpt_path and os.path.exists(ckpt_path):
    model, cfg = load_model_from_checkpoint(ckpt_path, device)
    tokenizer  = load_tokenizer('tokenizer/tokenizer.json')

    test_prompts = [
        'Write a Python function to check if a number is prime.',
        'What is the difference between a stack and a queue?',
        'Write a SQL query to find the top 3 most expensive products.',
    ]

    for prompt in test_prompts:
        print(f'\n{"─"*55}')
        print(f'User: {prompt}')
        response = chat_turn(model, tokenizer, prompt, device=device, max_new_tokens=200)
        print(f'Aarohan:\n{response}')

    print('\n✅ Aarohan-350M quality check complete!')
else:
    print('No SFT checkpoint found — check training completed successfully')

In [ ]:
# ── Cell 10: Done! ────────────────────────────────────────────
import glob, os

print('SFT checkpoints saved to /kaggle/working/se-llm-350m/checkpoints_sft/')
for ckpt in sorted(glob.glob('checkpoints_sft/*.pt')):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'  {os.path.basename(ckpt):40s}  {size_mb:.0f} MB')

print()
print('✅ Aarohan-350M SFT complete!')
print('   Download sft_final.pt and test locally with:')
print('   python evaluation/generate.py --checkpoint checkpoints_sft/sft_final.pt --mode chat')